# Scheduler Simulation Dashboard

Interactive exploration notebook for `trace.json` files produced by the scheduler simulator.

## Setup

- Ensure the simulator has produced an up-to-date `trace.json` (default: `build/bin/Debug/results/trace.json`).
- This notebook assumes it is executed from the `python/` directory inside the repository.

In [1]:
pip install ipywidgets nbformat ipython plotly pandas kaleido

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_ROOT = PROJECT_ROOT / "results"
sys.path.append(str(NOTEBOOK_DIR))

import metrics_loader as ml

DEFAULT_TRACE = PROJECT_ROOT / "build" / "bin" / "Debug" / "results" / "trace.json"
trace_directory = DEFAULT_TRACE.parent

trace_files = sorted([path.name for path in trace_directory.glob("trace*.json")]) or [DEFAULT_TRACE.name]
trace_selector = widgets.Dropdown(options=trace_files, value=DEFAULT_TRACE.name, description="Trace:")
display(trace_selector)

trace = None
config = {}
summary = {}
tasks = None
ticks = None

Dropdown(description='Trace:', options=('trace.json', 'trace_fcfs.json', 'trace_mlfq.json', 'trace_rr.json', '…

In [3]:
selected_path = trace_directory / trace_selector.value
print(f'Using trace: {selected_path}')

trace = ml.load_trace(selected_path)
config = ml.config_dict(trace)
summary = ml.summary_dict(trace)

if selected_path.name == 'trace.json':
    figure_output_dir = RESULTS_ROOT
else:
    stem = selected_path.stem
    if stem.startswith('trace_'):
        scheduler_name = stem[len('trace_'):]
        figure_output_dir = RESULTS_ROOT / scheduler_name
    else:
        figure_output_dir = RESULTS_ROOT / stem
figure_output_dir.mkdir(parents=True, exist_ok=True)
print(f'Saving figures to: {figure_output_dir}')

summary

Using trace: o:\Projects\Emebedded-Systems-CPU-Scheduler-Simulator\build\bin\Debug\results\trace_mlfq.json
Saving figures to: o:\Projects\Emebedded-Systems-CPU-Scheduler-Simulator\results\mlfq


{'average_response_time_ms': 0.31383684909278653,
 'average_runtime_ms': 2.5133102278593022,
 'average_turnaround_time_ms': 3.6173932285903962,
 'average_wait_time_ms': 1.1030897426443478,
 'completed_tasks': 13557,
 'core_idle_time_ms': [11349, 17342, 23496, 27291],
 'cpu_utilization': {'average': 0.324925, 'samples': 30000},
 'simulation_end_ms': 30001,
 'simulation_start_ms': 0,
 'task_count': 13566,
 'total_idle_time_ms': 79478,
 'total_simulation_time_ms': 30001}

In [4]:
config

{'base_power_watts': 6.5,
 'clock_speed_mhz': 1800.0,
 'context_switch_cost_us': 25,
 'idle_power_watts': 1.2,
 'io_completion_quantum_us': 200,
 'max_power_watts': 15.0,
 'num_cores': 4,
 'planned_run_duration_ms': 30000,
 'system_name': '4 Core IoT Compute Node',
 'tick_interval_us': 1000,
 'verbose_logging': True}

In [5]:
import pandas as pd

tasks = ml.task_lifecycle_df(trace)
ticks = ml.ticks_df(trace)

tasks.head()

,arrival_ms,class,completion_ms,deadline_ms,dispatch_count,final_state,first_dispatch_ms,name,priority,requested_exec_max_ms,requested_exec_min_ms,runtime_ms,task_id,turnaround_time_ms,wait_time_ms
0,18690,rt,18692,10,1,completed,18690,wake_word_detection#1870,1,2,2,2,1869,2.0,0
1,12890,rt,12891,5,1,completed,12890,audio_processing#2579,2,1,1,1,5579,1.0,0
2,3625,rt,3627,5,1,completed,3625,audio_processing#726,2,2,2,2,3726,2.0,0
3,0,rt,2,10,1,completed,0,wake_word_detection#1,1,2,2,2,0,2.0,0
4,20757,interactive,20761,33,1,completed,20757,screen_refresh#630,7,4,4,4,13153,4.0,0


## Core schedule

In [6]:
core_fig = ml.make_core_timeline_figure(trace)
core_fig.write_html(figure_output_dir / 'core_timeline.html')
core_fig.write_image(figure_output_dir / 'core_timeline.png')
core_fig

## CPU utilisation over time

In [7]:
cpu_fig = ml.make_cpu_utilisation_figure(trace, rolling_window=200)
cpu_fig.write_html(figure_output_dir / 'cpu_utilisation.html')
cpu_fig.write_image(figure_output_dir / 'cpu_utilisation.png')
cpu_fig

## Core idle totals

In [8]:
idle_fig = ml.make_core_idle_bar_figure(trace)
idle_fig.write_html(figure_output_dir / 'core_idle_totals.html')
idle_fig.write_image(figure_output_dir / 'core_idle_totals.png')
idle_fig

## Task runtime vs wait

In [9]:
runtime_fig = ml.make_task_runtime_scatter(trace)
runtime_fig.write_html(figure_output_dir / 'task_runtime_scatter.html')
runtime_fig.write_image(figure_output_dir / 'task_runtime_scatter.png')
runtime_fig

## Additional exploration

In [10]:
tasks.groupby("class")["runtime_ms"].sum().sort_values(ascending=False)

class
rt             28586
interactive     5044
background       453
Name: runtime_ms, dtype: int64